# 10 · OpenStreetMap schools — all India (Geofabrik extracts)

**Source:** OpenStreetMap contributors, India extracts from Geofabrik (`download.geofabrik.de/asia/india/`,
six zones, updated daily). **Licence:** ODbL 1.0 — "© OpenStreetMap contributors".

**Why:** UDISE school coordinates outside Kerala are corrected automatically (plan section B). The most trusted
correction is an OSM school with nearly the same name close to the school (Kerala calibration: OSM-matched
schools more than 1 km off fell from 15 % to 6 %).

**Why Geofabrik, not Overpass:** on 2026-11-07 the public Overpass servers were overloaded (a tiny query took
73 s; the all-India school queries timed out), and Overpass asks large downloads to use extracts instead.

**Method:** each zone file (110–560 MB) is downloaded, every `amenity=school` is read with the GDAL OSM driver
(points as they are; school areas by a point inside them), the schools are saved as
`data/raw/osm/india/osm_schools_<zone>_<date>.json` (same layout as an Overpass answer) and the zone file is
deleted to save disk. A zone already saved is skipped, so the notebook can be rerun after an interruption.

## Imports and settings

In [1]:
# Project folder. Laptop: open the notebook from "PROJECTS IPYNB" (or the project root).
# Google Colab: upload the project folder to Google Drive ("My Drive") and set its name below.
from pathlib import Path
import sys
PROJECT_FOLDER = "Distance-Capacity-and-Inequality-in-Access-to-Government-and-Aided-Schools"  # Colab only: folder name inside My Drive
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive") / PROJECT_FOLDER
else:
    PROJECT = Path.cwd().parent if Path.cwd().name in ("notebooks", "PROJECTS IPYNB") else Path.cwd()
assert (PROJECT / "scripts").is_dir(), f"project folder not found: {PROJECT}"
sys.path.insert(0, str(PROJECT / "scripts"))
import datetime as dt
import json
import re
import shutil

import pyogrio
import requests

OUT_DIR = PROJECT / "data" / "raw" / "osm" / "india"
TMP_DIR = (Path("/content/tmp") if IN_COLAB else PROJECT / "data" / "tmp") / "geofabrik"  # Colab: fast local disk, not Drive
URL = "https://download.geofabrik.de/asia/india/{}-zone-latest.osm.pbf"
ZONES = ["north-eastern", "eastern", "western", "northern", "central", "southern"]
HEADERS = {"User-Agent": "msc-spatial-inequality-study/0.1 (academic project)"}

In [ ]:
# Colab only: install packages that Colab does not have by default (does nothing on the laptop)
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rapidfuzz", "pymupdf", "pyogrio"], check=True)

### `download()`

In [2]:
def download(zone):
    """Stream one Geofabrik zone extract to a temporary file (deleted after the schools are taken out)."""
    TMP_DIR.mkdir(parents=True, exist_ok=True)
    f = TMP_DIR / f"{zone}-zone-latest.osm.pbf"
    if f.exists():  # only complete downloads get this name (see .part below)
        return f
    part = f.with_suffix(".part")
    with requests.get(URL.format(zone), headers=HEADERS, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(part, "wb") as out:
            shutil.copyfileobj(r.raw, out, length=1 << 20)
    part.rename(f)
    print(f"  downloaded {f.name}: {f.stat().st_size / 1e6:.0f} MB")
    return f

### `tag()`

In [3]:
def tag(other_tags, key):
    """Value of one key in GDAL's other_tags text ('"key"=>"value",...')."""
    if not isinstance(other_tags, str):  # missing tags come back as NaN
        return None
    m = re.search(r'"' + re.escape(key) + r'"=>"((?:[^"\\]|\\.)*)"', other_tags)
    return m.group(1) if m else None

### `extract_schools()`

In [4]:
def extract_schools(pbf, zone):
    """amenity=school from the OSM file: points as they are, areas by a point inside them (GDAL OSM driver)."""
    s = lambda v: v if isinstance(v, str) and v else None  # GDAL gives NaN for missing values
    els = []
    pts = pyogrio.read_dataframe(pbf, layer="points", columns=["osm_id", "name", "other_tags"],
                                 where="other_tags LIKE '%\"amenity\"=>\"school\"%'")
    for i, name, other, g in zip(pts["osm_id"], pts["name"], pts["other_tags"], pts.geometry):
        els.append({"type": "node", "id": int(i), "lat": g.y, "lon": g.x,
                    "tags": {k: v for k, v in [("name", s(name)), ("name:en", tag(other, "name:en"))] if v}})
    polys = pyogrio.read_dataframe(pbf, layer="multipolygons",
                                   columns=["osm_id", "osm_way_id", "name", "amenity", "other_tags"],
                                   where="amenity = 'school'")
    for rid, wid, name, other, g in zip(polys["osm_id"], polys["osm_way_id"], polys["name"],
                                        polys["other_tags"], polys.geometry):
        p = g.representative_point()
        els.append({"type": "relation" if s(rid) else "way", "id": int(s(rid) or wid), "center": {"lat": p.y, "lon": p.x},
                    "tags": {k: v for k, v in [("name", s(name)), ("name:en", tag(other, "name:en"))] if v}})
    return {"source": URL.format(zone), "generator": "GDAL OSM driver (pyogrio)", "elements": els}

### `run_all()`

In [5]:
def run_all():
    """One output per zone: osm_schools_<zone>_<date>.json. A zone already saved (any date) is skipped."""
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    for zone in ZONES:
        if list(OUT_DIR.glob(f"osm_schools_{zone}_*.json")):
            print(zone, "already saved")
            continue
        pbf = download(zone)
        d = extract_schools(pbf, zone)
        f = OUT_DIR / f"osm_schools_{zone}_{dt.date.today().isoformat()}.json"
        f.write_text(json.dumps(d), encoding="utf-8")
        pbf.unlink()  # keep only the schools, to save disk
        print(f"{zone}: {len(d['elements'])} schools -> {f.name}")

## Run

Six downloads (1.7 GB in total, one at a time) and extraction; roughly 20–60 minutes.

In [6]:
run_all()

north-eastern: 749 schools -> osm_schools_north-eastern_2026-11-07.json


  downloaded eastern-zone-latest.osm.pbf: 248 MB


C:\Users\ASUS\miniforge3\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: Non closed ring detected. To avoid accepting it, set the OGR_GEOMETRY_ACCEPT_UNCLOSED_RING configuration option to NO
  return ogr_read(


eastern: 2156 schools -> osm_schools_eastern_2026-11-07.json


  downloaded western-zone-latest.osm.pbf: 222 MB


western: 3040 schools -> osm_schools_western_2026-11-07.json


... (4 outputs left out)

central: 2796 schools -> osm_schools_central_2026-11-07.json


  downloaded southern-zone-latest.osm.pbf: 559 MB


C:\Users\ASUS\miniforge3\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: Non closed ring detected. To avoid accepting it, set the OGR_GEOMETRY_ACCEPT_UNCLOSED_RING configuration option to NO
  return ogr_read(


southern: 25524 schools -> osm_schools_southern_2026-11-07.json


## Check

In [7]:
tot = 0
for f in sorted(OUT_DIR.glob("osm_schools_*.json")):
    els = json.loads(f.read_text(encoding="utf-8"))["elements"]
    named = sum(1 for e in els if e.get("tags"))
    tot += len(els)
    print(f"{f.name:50s} {len(els):7d} schools, {named:7d} named")
print("total:", tot)

osm_schools_central_2026-11-07.json                   2796 schools,    2318 named
osm_schools_eastern_2026-11-07.json                   2156 schools,    1845 named
osm_schools_north-eastern_2026-11-07.json              749 schools,     644 named


osm_schools_northern_2026-11-07.json                  4217 schools,    3175 named
osm_schools_southern_2026-11-07.json                 25524 schools,   23131 named


osm_schools_western_2026-11-07.json                   3040 schools,    2610 named
total: 38482
